# IMDb Web Scraper: Advanced Search & Data Extraction

This project demonstrates an automated web scraping pipeline using **Selenium** to navigate complex dynamic filters and bypass lazy loading, combined with **BeautifulSoup** to parse and extract the HTML structure. 

The goal is to extract key movie metadata (titles, release years, durations, and ratings) and export it into a clean, structured format (Excel) for further data analysis.

# 1. Library Preparation & Browser Initialization
This stage loads all the required libraries (Selenium, BeautifulSoup, Pandas) and opens the Chrome browser in undetected mode. This mode is crucial for bypassing IMDb's initial anti-bot protection.

In [3]:
# SEL 1
import time
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import requests

# ---------------------------------------------------------
# 1. BUKA BROWSER & MENUJU ADVANCED SEARCH
# ---------------------------------------------------------
print("Membuka browser...")
driver = uc.Chrome(version_main=152)
driver.get('https://www.imdb.com/')
time.sleep(8)
print("Berhasil! Judul web:", driver.title)

Membuka browser...
Berhasil! Judul web: IMDb: Ratings, Reviews, and Where to Watch the Best Movies & TV Shows


# 2. Navigation & Filling Advanced Search Filters
Automating human interaction on the IMDb search form. This script automatically selects the title type, year range, rating limits, Oscar nominations, and handles the language input dropdown using keyboard stroke simulation.

In [4]:
# # Buka Dropdown kategori
dropdown = driver.find_element(By.CSS_SELECTOR, '[data-testid="category-selector-button"]')
dropdown.click()
time.sleep(2)

# Klik link Advanced Search
element = driver.find_element(By.XPATH, "//a[contains(., 'Advanced search')]")
driver.execute_script("arguments[0].click();", element)
time.sleep(5)
print("Sudah berada di halaman Advanced Search!")


# ---------------------------------------------------------
# 2. FILTER TITLE TYPE (MOVIE & TV SERIES)
# ---------------------------------------------------------
movie = driver.find_element(By.CSS_SELECTOR, '[data-testid="test-chip-id-movie"]')
driver.execute_script("arguments[0].click();", movie)
time.sleep(1)

tv_series = driver.find_element(By.CSS_SELECTOR, '[data-testid="test-chip-id-tvSeries"]')
driver.execute_script("arguments[0].click();", tv_series)
time.sleep(1)


# ---------------------------------------------------------
# 3. FILTER RELEASE DATE
# ---------------------------------------------------------
btn_release = driver.find_element(By.XPATH, "//*[text()='Release date']")
driver.execute_script("arguments[0].scrollIntoView(true);", btn_release)
time.sleep(1)
driver.execute_script("arguments[0].click();", btn_release)
time.sleep(1)

start_year = driver.find_element(By.CSS_SELECTOR, '[data-testid="releaseYearMonth-start"]')
start_year.clear()
start_year.send_keys("1996")

end_year = driver.find_element(By.CSS_SELECTOR, '[data-testid="releaseYearMonth-end"]')
end_year.clear()
end_year.send_keys("2026")
time.sleep(2)


# ---------------------------------------------------------
# 4. FILTER IMDB RATINGS
# ---------------------------------------------------------
btn_rating = driver.find_element(By.XPATH, "//*[text()='IMDb ratings']")
driver.execute_script("arguments[0].click();", btn_rating)
time.sleep(1)

min_rating = driver.find_element(By.CSS_SELECTOR, '[data-testid="imdbratings-start"]')
min_rating.send_keys("1.0")

max_rating = driver.find_element(By.CSS_SELECTOR, '[data-testid="imdbratings-end"]')
max_rating.send_keys("10.0")
time.sleep(1)


# ---------------------------------------------------------
# 5. FILTER AWARDS & RECOGNITION
# ---------------------------------------------------------
btn_awards = driver.find_element(By.XPATH, "//*[text()='Awards & recognition']")
driver.execute_script("arguments[0].click();", btn_awards)
time.sleep(1)

oscar_button = driver.find_element(By.CSS_SELECTOR, '[data-testid="test-chip-id-oscar-nominated"]')
driver.execute_script("arguments[0].click();", oscar_button)
time.sleep(1)


# ---------------------------------------------------------
# 6. FILTER COLOR INFO
# ---------------------------------------------------------
btn_color = driver.find_element(By.XPATH, "//*[text()='Color info']")
driver.execute_script("arguments[0].click();", btn_color)
time.sleep(1)

color_button = driver.find_element(By.CSS_SELECTOR, '[data-testid="test-chip-id-COLOR"]')
driver.execute_script("arguments[0].click();", color_button)
time.sleep(1)


# ---------------------------------------------------------
# 7. FILTER LANGUAGES (DENGAN SIMULASI KEYBOARD)
# ---------------------------------------------------------
btn_lang = driver.find_element(By.XPATH, "//*[text()='Languages']")
driver.execute_script("arguments[0].scrollIntoView(true);", btn_lang)
time.sleep(1)
driver.execute_script("arguments[0].click();", btn_lang)
time.sleep(1)

lang_input = driver.find_element(By.CSS_SELECTOR, '[data-testid="autosuggest-input-test-id-languages"]')
lang_input.clear()
lang_input.send_keys("English")
time.sleep(2) # Tunggu opsi muncul di bawah kotak

# Meniru manusia: Tekan Panah Bawah lalu ENTER di kotak bahasa
lang_input.send_keys(Keys.ARROW_DOWN)
time.sleep(1)
lang_input.send_keys(Keys.ENTER)

print("Bahasa dipilih! Menunggu tombol See results menyala...")
time.sleep(3) # Jeda penting agar sistem IMDb mengubah tombol abu-abu menjadi kuning


# ---------------------------------------------------------
# 8. SUBMIT (SEE RESULTS)
# ---------------------------------------------------------
print("Mengeksekusi tombol See results...")

# Karena tombol sudah aktif (kuning), JavaScript murni akan berhasil mengekliknya
driver.execute_script("""
    let buttons = document.querySelectorAll('[data-testid="adv-search-get-results"]');
    buttons.forEach(btn => {
        if (!btn.disabled) { // Hanya klik tombol yang sudah aktif/kuning
            btn.click();
        }
    });
""")

time.sleep(8)
print("Selesai! Halaman daftar film siap untuk diskraping.")


#############################################

current_url = driver.current_url
print("Menuju URL:", current_url)

# 1. Buat sebuah Sesi (Session) agar pengaturan topeng (User-Agent) dan Cookie tersimpan
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9'
})

# 2. Ambil semua cookie dari Selenium, lalu masukkan ke dalam sesi requests
for cookie in driver.get_cookies():
    session.cookies.set(cookie['name'], cookie['value'])

# 3. Eksekusi permintaan dengan sesi yang sudah dipersenjatai Cookie
response = session.get(current_url)
print("Status Unduhan:", response.status_code) 

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    print("BeautifulSoup siap digunakan!")
else:
    print("Gagal menembus. Disarankan kembali menggunakan Solusi 1.")

Sudah berada di halaman Advanced Search!
Bahasa dipilih! Menunggu tombol See results menyala...
Mengeksekusi tombol See results...
Selesai! Halaman daftar film siap untuk diskraping.
Menuju URL: https://www.imdb.com/search/title/?title_type=feature,tv_series&release_date=1996-01-01,2026-12-31&user_rating=1,10&groups=oscar_nominee&colors=color&languages=en
Status Unduhan: 200
BeautifulSoup siap digunakan!


# 3. Bypassing Lazy Loading (Auto-Scroll & Load More)
IMDb limits the page display to save server load. This cell uses a loop to automatically scroll down the page and continuously click the "50 more" button until the entire dataset (1,000+ movies) is fully loaded on the screen.

In [5]:
############# SEL 3 ###############
import pandas as pd

# Siapkan tempat penampungan (list)
movie_title = []
year = []
duration = []
rating = []

print("Mulai mengekstrak data...")

# Tambahkan dua baris ini di paling atas Sel 3
list_items = soup.find_all('li', class_='ipc-metadata-list-summary-item')
print(f"Total film awal yang tertangkap: {len(list_items)}")

# Ekstrak data satu per satu
for result in list_items:
    # 1. Mengambil Judul (menggunakan h4)
    title_elem = result.find('h4', class_='ipc-title__text')
    movie_title.append(title_elem.text if title_elem else "N/A")
    
    # 2. Mengambil Tahun dan Durasi (karena class-nya sama, kita pakai find_all)
    metadata = result.find_all('li', class_='ipc-inline-list__item')
    
    if len(metadata) >= 2:
        year.append(metadata[0].text)        # Urutan pertama biasanya Tahun
        duration.append(metadata[1].text)    # Urutan kedua biasanya Durasi
    elif len(metadata) == 1:
        year.append(metadata[0].text)
        duration.append("N/A")
    else:
        year.append("N/A")
        duration.append("N/A")
        
    # 3. Mengambil Rating
    rating_elem = result.find('span', class_='ipc-rating-star--rating')
    rating.append(rating_elem.text if rating_elem else "N/A")

# Gabungkan ke dalam Pandas DataFrame
imdb_df = pd.DataFrame({
    'Movie Title': movie_title,
    'Year': year,
    'Duration': duration,
    'Rating': rating
})

print("Ekstraksi selesai! Berikut hasilnya:")
# Tampilkan 10 baris pertama
print(imdb_df.head(50))

Mulai mengekstrak data...
Total film awal yang tertangkap: 25
Ekstraksi selesai! Berikut hasilnya:
                                          Movie Title  Year Duration Rating
0                                1. Avengers: Endgame  2019    3h 1m    8.4
1                                        2. United 93  2006   1h 51m    7.6
2                                     3. Interstellar  2014   2h 49m    8.7
3                                  4. The Sixth Sense  1999   1h 47m    8.2
4                                          5. Weapons  2025    2h 8m    7.4
5                                  6. The Dark Knight  2008   2h 32m    9.1
6                                        7. Inception  2010   2h 28m    8.8
7                                             8. Troy  2004   2h 43m    7.3
8                                            9. Anora  2024   2h 19m    7.4
9                                    10. The Prestige  2006   2h 10m    8.5
10                                        11. Sinners  2025   2h 

# 4. Extracting Raw HTML (Page Source)
Once all elements are successfully loaded visually by Selenium, we extract the entire HTML code of the page at once. This raw HTML is then passed to BeautifulSoup to be parsed based on the movie element containers.

In [6]:
################ SEL 4 ############
import time
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

print("Mulai menggulir untuk memuat SELURUH film (mohon bersabar, ini memakan waktu sekitar 1-2 menit)...")

klik_berhasil = 0

# Menggunakan while True agar script berjalan terus sampai data habis
while True:
    try:
        # 1. Gulir perlahan ke bagian paling bawah halaman
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2) # Tunggu animasi gulir
        
        # 2. Cari tombol "Load More"
        load_more_btn = driver.find_element(By.CSS_SELECTOR, "button.ipc-see-more__button")
        
        # 3. Klik tombol menggunakan JavaScript
        driver.execute_script("arguments[0].click();", load_more_btn)
        
        # 4. Beri waktu agar 50 film tambahan termuat sempurna di layar (PENTING: Jangan terlalu cepat)
        time.sleep(4) 
        
        klik_berhasil += 1
        print(f"Berhasil memuat kloter ke-{klik_berhasil + 1}...")
        
    except Exception as e:
        # Jika tombol tidak ditemukan lagi (error), berarti kita sudah di ujung halaman (1.082 film)
        print("Tombol '50 more' sudah tidak ditemukan. Semua film berhasil dimuat!")
        break # Hentikan perulangan

print("-" * 30)
print(f"Selesai! Total klik tambahan yang dilakukan: {klik_berhasil} kali.")

# ---------------------------------------------------------
# TARIK HTML-NYA SETELAH SEMUA DATA TERBUKA
# ---------------------------------------------------------
html_mentah = driver.page_source
soup = BeautifulSoup(html_mentah, 'html.parser')

# Kita cek apakah jumlah bungkus filmnya sudah mencapai 1000+
list_items = soup.find_all('li', class_='ipc-metadata-list-summary-item')
print(f"Total keseluruhan film yang siap diekstrak: {len(list_items)}")

Mulai menggulir untuk memuat SELURUH film (mohon bersabar, ini memakan waktu sekitar 1-2 menit)...
Berhasil memuat kloter ke-2...
Berhasil memuat kloter ke-3...
Berhasil memuat kloter ke-4...
Berhasil memuat kloter ke-5...
Berhasil memuat kloter ke-6...
Berhasil memuat kloter ke-7...
Berhasil memuat kloter ke-8...
Berhasil memuat kloter ke-9...
Berhasil memuat kloter ke-10...
Berhasil memuat kloter ke-11...
Berhasil memuat kloter ke-12...
Berhasil memuat kloter ke-13...
Berhasil memuat kloter ke-14...
Berhasil memuat kloter ke-15...
Berhasil memuat kloter ke-16...
Berhasil memuat kloter ke-17...
Berhasil memuat kloter ke-18...
Berhasil memuat kloter ke-19...
Berhasil memuat kloter ke-20...
Berhasil memuat kloter ke-21...
Berhasil memuat kloter ke-22...
Tombol '50 more' sudah tidak ditemukan. Semua film berhasil dimuat!
------------------------------
Selesai! Total klik tambahan yang dilakukan: 21 kali.
Total keseluruhan film yang siap diekstrak: 1082


# 5. Data Cleaning & Extraction to Pandas DataFrame
Extracting specific information (Title, Year, Duration, and Rating) from each movie's HTML element. This cell also performs automated data cleaning to remove the ranking numbers from the titles before assembling them into a structured table (DataFrame).

In [7]:
import pandas as pd

# Siapkan tempat penampungan
movie_title = []
year = []
duration = []
rating = []

print("Mulai mengekstrak 200 data film...")

# Ekstrak data dari ke-200 list_items
for result in list_items:
    # 1. Mengambil Judul & Membersihkan Angka di depannya
    title_elem = result.find('h4', class_='ipc-title__text')
    if title_elem:
        # Memecah teks berdasarkan titik pertama, lalu mengambil bagian kanannya
        # "1. Avengers: Endgame" -> menjadi -> "Avengers: Endgame"
        clean_title = title_elem.text.split('.', 1)[-1].strip()
        movie_title.append(clean_title)
    else:
        movie_title.append("N/A")
        
    # 2. Mengambil Tahun dan Durasi
    metadata = result.find_all('li', class_='ipc-inline-list__item')
    
    if len(metadata) >= 2:
        year.append(metadata[0].text)        
        duration.append(metadata[1].text)    
    elif len(metadata) == 1:
        year.append(metadata[0].text)
        duration.append("N/A")
    else:
        year.append("N/A")
        duration.append("N/A")
        
    # 3. Mengambil Rating
    rating_elem = result.find('span', class_='ipc-rating-star--rating')
    rating.append(rating_elem.text if rating_elem else "N/A")

# Gabungkan ke dalam Pandas DataFrame
imdb_df = pd.DataFrame({
    'Movie Title': movie_title,
    'Year': year,
    'Duration': duration,
    'Rating': rating
})

print("Ekstraksi selesai! Berikut 5 baris pertama dan terakhir dari tabelmu:")
print("-" * 50)
print(imdb_df)

Mulai mengekstrak 200 data film...
Ekstraksi selesai! Berikut 5 baris pertama dan terakhir dari tabelmu:
--------------------------------------------------
                                            Movie Title  Year Duration Rating
0                                     Avengers: Endgame  2019    3h 1m    8.4
1                                             United 93  2006   1h 51m    7.6
2                                          Interstellar  2014   2h 49m    8.7
3                                       The Sixth Sense  1999   1h 47m    8.2
4                                               Weapons  2025    2h 8m    7.4
...                                                 ...   ...      ...    ...
1077    On Tiptoe: The Music of Ladysmith Black Mambazo  2000      58m    7.5
1078  Tell the Truth and Run: George Seldes and the ...  1996   1h 30m    7.7
1079                                   Regret to Inform  1998   1h 12m    7.2
1080                                Speaking in Strings  1999   

# 6. Exporting Final Results to Excel (.xlsx)
Saving the clean, final results into a local folder. This Excel file is ready to be delivered as the final output (deliverables) or used further for data visualization and analysis.

In [8]:
import os

# 1. Membuat folder bernama 'data' jika belum ada (agar rapi)
# Ini adalah praktik standar dalam menyusun struktur folder proyek
if not os.path.exists('data'):
    os.makedirs('data')

# 2. Menentukan nama file dan lokasi simpannya
nama_file = 'data/imdb_advanced_search_results.xlsx'

# 3. Mengekspor DataFrame Pandas ke dalam file Excel
# index=False digunakan agar nomor urut bawaan Pandas (0, 1, 2...) tidak ikut masuk ke Excel
imdb_df.to_excel(nama_file, index=False)

print(f"Sukses besar! Data 1.082 film telah berhasil disimpan di dalam file: {nama_file}")

Sukses besar! Data 1.082 film telah berhasil disimpan di dalam file: data/imdb_advanced_search_results.xlsx
